In [ ]:
#| default_exp spec

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

What a kernel is started from, and the shape of what it returns.

Nothing here starts a kernel or talks to one. `_spec` builds the kernelspec, `bootstrap_src` builds
the source of the cell that runs first inside a kernel, and `ExecOutcome` is what one
`execute_request` comes back as. `kunda.kernel` does the starting.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import importlib.util, os, subprocess, sys

In [ ]:
#| export
from dataclasses import dataclass, field

In [ ]:
#| export
from pathlib import Path

In [ ]:
#| export
from fastcore.all import L, first

In [ ]:
#| export
from jupyter_client.kernelspec import KernelSpec

In [ ]:
#| export
from kunda.support import HOST_PY, clean_env, support_paths

In [ ]:
#| export
def run_file_src(path, src=None, argv=(), cwd=None):
    "Code that runs a file in the kernel's *own* namespace, with script semantics."
    src = open(path, encoding='utf-8').read() if src is None else src
    setup = ('import os as _kd_os, sys as _kd_sys\n'
        f'_kd_code = compile({src!r}, {str(path)!r}, "exec")\n'
        '_kd_argv, _kd_cwd = _kd_sys.argv, _kd_os.getcwd()\n'
        f'_kd_sys.argv = [{str(path)!r}, *{list(argv)!r}]\n')
    if cwd: setup += f'_kd_os.chdir({str(cwd)!r})\n'
    return setup + ('try: exec(_kd_code)\n'
        'finally:\n'
        '    _kd_sys.argv = _kd_argv\n'
        '    _kd_os.chdir(_kd_cwd)\n'
        '    del _kd_os, _kd_sys, _kd_code, _kd_argv, _kd_cwd\n')

`run_file_src` builds the source of a cell that runs a file. The kernel executes it in the namespace
it already has, so what the file defines is still there for the next cell, and what is already there
is visible to the file.

`sys.argv[0]` is `path` and `argv` supplies the rest. `cwd` applies for the length of the run. The
generated code restores `sys.argv` and the working directory in a `finally`, and deletes the names
it introduced, whether the file returned or raised.

`path` is compiled into the code object, so a traceback names the file. `src` overrides what is read
from it: an editor can run a buffer that was never saved and still get that traceback.

In [ ]:
print(run_file_src('/proj/train.py', src='import sys\nprint(sys.argv)\n', argv=['--epochs', '3']))

import os as _kd_os, sys as _kd_sys
_kd_code = compile('import sys\nprint(sys.argv)\n', '/proj/train.py', "exec")
_kd_argv, _kd_cwd = _kd_sys.argv, _kd_os.getcwd()
_kd_sys.argv = ['/proj/train.py', *['--epochs', '3']]
try: exec(_kd_code)
finally:
    _kd_sys.argv = _kd_argv
    _kd_os.chdir(_kd_cwd)
    del _kd_os, _kd_sys, _kd_code, _kd_argv, _kd_cwd



In [ ]:
#| hide
tmp = TemporaryDirectory(); d = Path(tmp.name)
(d/'greet.py').write_text('import os, sys\ngreeting = f"hello {sys.argv[1]}"\nran_in = os.getcwd()\n')
argv0, cwd0, ns = list(sys.argv), os.getcwd(), {}
exec(run_file_src(d/'greet.py', argv=['world'], cwd=d), ns)
test_eq(ns['greeting'], 'hello world')
test_eq(Path(ns['ran_in']).resolve(), d.resolve())
assert not [k for k in ns if k.startswith('_kd_')], 'the setup leaves nothing behind'
test_fail(lambda: exec(run_file_src(d/'greet.py', src='raise ValueError("boom")\n'), {}), contains='boom')
test_eq(sys.argv, argv0)
test_eq(os.getcwd(), cwd0)

In [ ]:
#| export
BOOTSTRAP = '''
def _kd_bootstrap():
	import sys, importlib.util
	# Borrowed only by a kernel of the host's own minor version. A frozen build ships bytecode-only
	# modules in its zip, and another version reads their magic number and refuses: an import that
	# fell through to them died as `bad magic number`, not as anything about the inspector.
	if tuple(sys.version_info[:2]) == tuple({version!r}):
		for _p in {support!r}:
			if _p and _p not in sys.path: sys.path.append(_p)
	# The project venv may contain an older dhrishti. Pin the inspector protocol to this
	# host installation while every other import stays project-local. Best-effort: a frozen
	# build imports dhrishti out of a zip, where there is no file to point at. A failed pin
	# must leave nothing in sys.modules, or the fallback import finds the broken module.
	try:
		if tuple(sys.version_info[:2]) != tuple({version!r}): raise ImportError('another Python')
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]:
			del sys.modules[_n]
		_spec = importlib.util.spec_from_file_location('dhrishti', {dhrishti_init!r},
			submodule_search_locations=[{dhrishti_dir!r}])
		_pkg = importlib.util.module_from_spec(_spec)
		sys.modules['dhrishti'] = _pkg
		_spec.loader.exec_module(_pkg)
	except Exception:
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]:
			del sys.modules[_n]
	import dhrishti.serving as ls
	r = ls.serve_in_kernel(name={name!r}, port={port!r}, agent={agent!r}, token={token!r},
	                       session_dir={sessions!r}, agent_session_dir={agent_sessions!r})
	if {ipymini!r}:
		try:
			from IPython import get_ipython
			ls.set_kernel_backend(getattr(get_ipython(), 'kernel', None))
		except Exception: ls.set_kernel_backend(None)
	return r
try: _kd_bootstrap()
finally: del _kd_bootstrap
'''

`BOOTSTRAP` is the template for the cell a kernel runs before anything of the user's.

Two rules are written into it, and both are guarded on the kernel's own `sys.version_info`. The
host's `sys.path` entries are appended only where the kernel's minor version matches the host's: a
frozen build ships bytecode-only modules, and another version reads their magic number and refuses.
The inspector is pinned to the host's own `dhrishti` under the same guard, so the protocol matches
whatever is asking. Every other import stays project-local.

A pin that fails leaves nothing named `dhrishti` in `sys.modules`, and the plain import that follows
finds the project's own copy rather than a half-loaded one.

The bootstrap defines one function and deletes it in a `finally`. The kernel's namespace keeps
neither it nor anything it imported.

In [ ]:
args = dict(name='nb-1', port=8123, agent='restricted', token=True, support=['/host/site-packages'],
            sessions='_kunda_sessions', agent_sessions='_kunda_agent_sessions', version=(3, 12),
            dhrishti_init='/host/dhrishti/__init__.py', dhrishti_dir='/host/dhrishti', ipymini=False)
src = BOOTSTRAP.format(**args)
print(src)


def _kd_bootstrap():
	import sys, importlib.util
	# Borrowed only by a kernel of the host's own minor version. A frozen build ships bytecode-only
	# modules in its zip, and another version reads their magic number and refuses: an import that
	# fell through to them died as `bad magic number`, not as anything about the inspector.
	if tuple(sys.version_info[:2]) == tuple((3, 12)):
		for _p in ['/host/site-packages']:
			if _p and _p not in sys.path: sys.path.append(_p)
	# The project venv may contain an older dhrishti. Pin the inspector protocol to this
	# host installation while every other import stays project-local. Best-effort: a frozen
	# build imports dhrishti out of a zip, where there is no file to point at. A failed pin
	# must leave nothing in sys.modules, or the fallback import finds the broken module.
	try:
		if tuple(sys.version_info[:2]) != tuple((3, 12)): raise ImportError('another Python')
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')

In [ ]:
#| hide
assert compile(src, '<bootstrap>', 'exec'), 'generated source, so it has to parse'
test_eq(src.count('(3, 12)'), 2)                  # one guard for the path, one for the pin
assert 'if False:' in src
mini = BOOTSTRAP.format(**{**args, 'ipymini': True})
assert compile(mini, '<bootstrap>', 'exec')
assert 'if True:' in mini

In [ ]:
#| export
def bootstrap_src(name=None, port=8000, agent='restricted', token=True, kernel='ipykernel',
                  sessions='_kunda_sessions', agent_sessions='_kunda_agent_sessions'):
    "The bootstrap cell source for a kernel that should host an inspector."
    support = support_paths()
    # Found, not imported: importing dhrishti for its `__file__` drags IPython, pandas and
    # numpy into the host process, which has no use for them. The kernel does the importing.
    spec = importlib.util.find_spec('dhrishti')
    if spec is None or not spec.origin: raise RuntimeError('dhrishti is not installed')
    dhrishti_init = str(Path(spec.origin).resolve())
    return BOOTSTRAP.format(name=name, port=port, agent=agent, token=token, support=support,
        sessions=sessions, agent_sessions=agent_sessions,
        version=HOST_PY, dhrishti_init=dhrishti_init,
        dhrishti_dir=str(Path(dhrishti_init).parent), ipymini=(kernel == 'ipymini'))

`bootstrap_src` supplies those values. `support_paths` gives the host's `sys.path`, `HOST_PY` the
version both guards compare against, and `importlib.util.find_spec` the file the host's `dhrishti`
package was loaded from.

The spec is found, not imported. Importing `dhrishti` for its `__file__` would drag IPython, pandas
and numpy into the host process, which has no use for them.

A host with no `dhrishti` installed raises `RuntimeError` rather than returning a bootstrap that
cannot work.

In [ ]:
#| hide
if importlib.util.find_spec('dhrishti') is None: test_fail(bootstrap_src, contains='dhrishti is not installed')
else: assert compile(bootstrap_src(name='nb-1'), '<bootstrap>', 'exec')

In [ ]:
#| export
def output_text(outs):
    "Flatten nbformat outputs to plain text: the terminal rendering, and the agent's view of a run."
    parts = L()
    for o in outs:
        t = o.get('output_type')
        if t == 'stream': parts.append(o.get('text', ''))
        elif t == 'error': parts.append('\n'.join(o.get('traceback') or [f"{o.get('ename')}: {o.get('evalue')}"]))
        elif t in ('execute_result', 'display_data'):
            d = o.get('data') or {}
            parts.append(d.get('text/markdown') or d.get('text/plain') or next((f'[{k}]' for k in d), ''))
    return ''.join(parts)

`output_text` flattens nbformat outputs the way a terminal would show them. Streams and error
tracebacks go through as they are. A result or a display takes `text/markdown` first, then
`text/plain`, and where it has neither it contributes the name of its first mime type in brackets,
so an image is accounted for in the text without being rendered.

An error carrying no `traceback` falls back to `ename: evalue`. Any other `output_type` contributes
nothing.

In [ ]:
outs = [{'output_type': 'stream', 'name': 'stdout', 'text': 'fitting\n'},
        {'output_type': 'execute_result', 'data': {'text/plain': '0.94'}},
        {'output_type': 'display_data', 'data': {'image/png': 'iVBORw0KGgo'}}]
print(output_text(outs))

fitting
0.94[image/png]


In [ ]:
#| hide
test_eq(output_text([]), '')
test_eq(output_text([{'output_type': 'error', 'ename': 'ValueError', 'evalue': 'boom'}]), 'ValueError: boom')
test_eq(output_text([{'output_type': 'error', 'traceback': ['Traceback', 'ValueError: boom']}]),
        'Traceback\nValueError: boom')
test_eq(output_text([{'output_type': 'display_data', 'data': {'text/markdown': '**hi**', 'text/plain': 'hi'}}]),
        '**hi**')
test_eq(output_text([{'output_type': 'execute_result', 'data': {}}]), '')
test_eq(output_text([{'output_type': 'update_display_data', 'data': {'text/plain': 'x'}}]), '')

In [ ]:
#| export
@dataclass
class ExecOutcome:
    "Result of one execute_request: nbformat-shaped outputs plus the shell reply status."
    ok: bool = True
    execution_count: int | None = None
    outputs: list = field(default_factory=list)
    error: str | None = None
    @property
    def text(self): return output_text(self.outputs)

`ExecOutcome` is one `execute_request` answered. A cell that raises is still an outcome: `ok` is
False, `error` holds the reply's error line, and `text` holds the traceback the way a terminal would
show it. The caller sees no exception.

`outputs` is nbformat, so it can be written into a notebook unchanged. `text` renders it on demand
and is not stored.

In [ ]:
out = ExecOutcome(execution_count=3, outputs=[{'output_type': 'stream', 'text': 'fitting\n'}])
out.ok, out.execution_count, out.text

(True, 3, 'fitting\n')

In [ ]:
#| hide
test_eq(ExecOutcome().text, '')
bad = ExecOutcome(ok=False, error='ValueError: boom',
                  outputs=[{'output_type': 'error', 'traceback': ['ValueError: boom']}])
test_eq((bad.ok, bad.text), (False, 'ValueError: boom'))

In [ ]:
#| export
def _runtime_python(python=None):
    "A selected interpreter, or py2app's bundled generic Python helper."
    if python: return str(python)
    if getattr(sys, 'frozen', False):
        helper = Path(sys.executable).with_name('python')
        if helper.exists(): return str(helper)
    return sys.executable

`_runtime_python` answers what goes at the front of a kernel's argv. A selected interpreter is used
as given. With none, a frozen build takes the generic `python` helper py2app leaves beside the app's
own executable, since that executable is the app and cannot be asked to run `-m ipykernel_launcher`.
Otherwise it is this interpreter.

In [ ]:
_runtime_python('/repo/.venv/bin/python'), _runtime_python() == sys.executable

('/repo/.venv/bin/python', True)

In [ ]:
#| export
class KernelStartError(RuntimeError):
    "A kernel that could not start, said in terms of the environment rather than the protocol."

In [ ]:
#| export
def missing_kernel_module(python=None, kernel='ipykernel'):
    """The kernel package `python` cannot import, or None.

    A venv without it exits before the first message, and jupyter_client can only report that the
    kernel died before replying to `kernel_info`. Asked here, the environment says which module it
    is short of. Only the failure path pays for this.
    """
    mod = 'ipymini' if kernel == 'ipymini' else 'ipykernel_launcher'
    exe = _runtime_python(python)
    if exe == sys.executable: return None if importlib.util.find_spec(mod) else mod
    src = f"import importlib.util, sys; sys.exit(0 if importlib.util.find_spec({mod!r}) else 1)"
    try: r = subprocess.run([exe, '-c', src], capture_output=True, timeout=20, env=clean_env())
    except (OSError, subprocess.SubprocessError): return None   # cannot tell, so do not say
    return None if r.returncode == 0 else mod

`missing_kernel_module` asks an environment which kernel package it is short of. It is asked only
after a start has failed, because `jupyter_client` can report no more than that the kernel died
before replying to `kernel_info`, which names nothing anyone can act on.

`None` is the answer both when the module imports and when the question could not be put at all. An
interpreter that will not run is no evidence about its packages, so nothing is claimed about it. The
import is tried in this process when the interpreter is this one, and in a subprocess with a
20 second timeout otherwise.

In [ ]:
missing_kernel_module(), missing_kernel_module(kernel='ipymini')

(None, None)

In [ ]:
#| hide
test_eq(missing_kernel_module('/no/such/python'), None)          # cannot tell, so does not say
if Path('/bin/false').exists():
    test_eq(missing_kernel_module('/bin/false'), 'ipykernel_launcher')
    test_eq(missing_kernel_module('/bin/false', 'ipymini'), 'ipymini')

In [ ]:
#| export
def _frozen_pythonpath():
    "Module and extension paths a py2app helper process must inherit."
    if not getattr(sys, 'frozen', False): return None
    resources = Path(sys.executable).resolve().parent.parent/'Resources'
    version = f'python{sys.version_info.major}.{sys.version_info.minor}'
    bundled = [resources/'lib'/version, resources/'lib'/version/'lib-dynload',
        resources/'lib'/f'python{sys.version_info.major}{sys.version_info.minor}.zip']
    return os.pathsep.join(dict.fromkeys(str(p) for p in [*bundled, *sys.path] if p))

In [ ]:
#| export
def _kernel_env(python=None):
    "Environment for a kernel child, detached from py2app's interpreter redirect."
    env = clean_env()
    if python is None and (path := _frozen_pythonpath()): env['PYTHONPATH'] = path
    return env

A kernel child is started under `clean_env`: this process's environment with a frozen host's
interpreter redirection taken out. Outside a bundle that is all of it, and `_frozen_pythonpath` is
`None`.

Inside one, a child on the bundled interpreter is given the bundle's own module and extension
directories on `PYTHONPATH`, which is the only way it finds them. A child on a chosen interpreter is
never given them. They are another version's bytecode, and the project's own environment is what it
is being started for.

In [ ]:
#| hide
test_eq(_frozen_pythonpath(), None)                              # this notebook is not a frozen build
test_eq(_kernel_env(), clean_env())
test_eq(_kernel_env('/repo/.venv/bin/python'), clean_env())

In [ ]:
#| export
def _spec(python=None, name='python3', kernel='ipykernel', lang='python', known=None, install=''):
    """A kernelspec pinned to a specific interpreter, so the kernel runs in *this* venv.

    Python is the language an interpreter is pinned for. Any other language runs whatever
    kernelspec is installed for it, unmodified, because there is no venv of ours to point it at."""
    if lang and lang != 'python': return installed_spec(lang, known, install)
    runtime = _runtime_python(python)
    if kernel == 'ipymini': argv = [runtime, '-Xfrozen_modules=off', '-m', 'ipymini', '-f', '{connection_file}']
    else: argv = [runtime, '-m', 'ipykernel_launcher', '-f', '{connection_file}']
    metadata = {'supported_encryption': 'curve'} if kernel == 'ipykernel' else {}
    return KernelSpec(argv=argv, display_name=name, language='python', metadata=metadata)

`_spec` builds a kernelspec whose `argv` names a chosen interpreter instead of looking one up by
kernelspec name. That is what puts the kernel in the project's own virtual environment.
`{connection_file}` is left in the argv for `jupyter_client` to substitute.

`language` is always `python`, whatever the interpreter, and `metadata` carries
`supported_encryption` for `ipykernel` only.

Any language other than `python` returns whatever kernelspec is installed for it, unmodified. There
is no environment of ours to point it at.

In [ ]:
s = _spec('/repo/.venv/bin/python', name='myrepo')
s.argv, s.metadata

(['/repo/.venv/bin/python',
  '-m',
  'ipykernel_launcher',
  '-f',
  '{connection_file}'],
 {'supported_encryption': 'curve'})

In [ ]:
_spec('/repo/.venv/bin/python', kernel='ipymini').argv

['/repo/.venv/bin/python',
 '-Xfrozen_modules=off',
 '-m',
 'ipymini',
 '-f',
 '{connection_file}']

In [ ]:
#| hide
test_eq((s.language, s.display_name), ('python', 'myrepo'))
test_eq(s.argv[-1], '{connection_file}')
test_eq(_spec().argv[0], sys.executable)                         # no interpreter named, this one
mini = _spec('/repo/.venv/bin/python', kernel='ipymini')
test_eq(mini.metadata, {})
assert '-Xfrozen_modules=off' in mini.argv and '-Xfrozen_modules=off' not in s.argv

In [ ]:
#| export
def installed_kernels():
    "Every Jupyter kernelspec on this machine, as `{name: spec}`. An unreadable store is none."
    from jupyter_client.kernelspec import KernelSpecManager
    try: return KernelSpecManager().get_all_specs()
    except Exception: return {}

`installed_kernels` is `KernelSpecManager().get_all_specs()`, with an unreadable or missing store
answering `{}` instead of raising. The keys are kernelspec names, and each value holds that spec's
own `language`.

In [ ]:
sorted(installed_kernels())

['ipymini', 'python3']

In [ ]:
#| export
def kernelspec_for(lang, known=None):
    """The installed kernelspec name that runs `lang`, or None.

    `known` is a caller's own `{language: kernelspec}` mapping, which wins where it answers: a host
    with a language registry knows which of several installed kernels it means.
    """
    specs = installed_kernels()
    if (name := (known or {}).get(str(lang))) and name in specs: return name
    return first(n for n, s in specs.items()
                 if str(((s.get('spec') or {}).get('language') or '')).lower() == str(lang).lower())

In [ ]:
#| hide
_real_installed = installed_kernels
def installed_kernels(): return {'python3': {'spec': {'language': 'python'}},
                                 'evcxr': {'spec': {'language': 'Rust'}},
                                 'xrust': {'spec': {'language': 'rust'}}}

`kernelspec_for` maps a language to an installed kernelspec name.

`known` is the caller's own `{language: kernelspec}` mapping and wins where it answers, so a host
with a language registry gets the kernel it means rather than the first that matches. A `known`
entry naming a kernelspec that is not installed is not an answer, and the search carries on.
Languages are compared case-insensitively. Nothing matching is `None`.

The examples below stand a fixed table of installed kernels in for this machine's: `evcxr` for
`Rust`, `xrust` for `rust`, and `python3`.

In [ ]:
kernelspec_for('rust'), kernelspec_for('rust', known={'rust': 'xrust'}), kernelspec_for('julia')

('evcxr', 'xrust', None)

In [ ]:
#| hide
test_eq(kernelspec_for('rust', known={'rust': 'not-installed'}), 'evcxr')   # a mapping to nothing is no answer
test_eq(kernelspec_for('python'), 'python3')
installed_kernels = _real_installed

In [ ]:
#| export
def installed_spec(lang, known=None, install=''):
    "The `KernelSpec` for `lang`, or a `KernelStartError` naming what would install one."
    from jupyter_client.kernelspec import KernelSpecManager
    if (name := kernelspec_for(lang, known)): return KernelSpecManager().get_kernel_spec(name)
    how = f' Install one with `{install}`.' if install else ''
    raise KernelStartError(f'no Jupyter kernel is installed for {lang}.{how}')

`installed_spec` is what a language other than `python` resolves to, through `_spec`. A language
with no kernelspec raises `KernelStartError`, and `install` puts the command that would fix it in
the message. Nobody can act on a missing kernel without being told which one is missing.

In [ ]:
try: installed_spec('rust', install='cargo install evcxr_jupyter')
except KernelStartError as e: print(e)

no Jupyter kernel is installed for rust. Install one with `cargo install evcxr_jupyter`.


In [ ]:
#| hide
test_fail(lambda: _spec(lang='rust'), exc=KernelStartError, contains='no Jupyter kernel is installed for rust')
if kernelspec_for('python'): test_eq(installed_spec('python').language, 'python')
tmp.cleanup()